# Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend

Task 4.1 — One validated call

In [ ]:
from google import genai
from google.genai import types
import os
import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError



In [ ]:
# 1. Define the JSON output schema
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

# 2. System instruction
SYSTEM_TEXT = (
    "You are an experienced healthcare operations analyst at CareLoop Clinics. "
    "Analyze patient follow-up call notes objectively, concisely, and accurately in a professional tone."
)

# 3. Connect to Gemini API (uses Colab Secrets if available, else prompts cleanly)
API_KEY = input("Paste your Gemini API key: ").strip().strip('"').strip("'")
client = genai.Client(api_key=API_KEY)


# 4. Build the CRAFT prompt cleanly
def build_craft_prompt(note_text: str) -> str:
  context = (
      "CareLoop Clinics calls patients who missed their outpatient appointment "
      "to understand why they did not show up."
  )
  role = "Healthcare operations analyst reviewing call-centre follow-up notes."
  action = """Read the follow-up note below and extract:
1. root_cause: choose the single best matching reason from this allowed list:
   - forgot_or_no_reminder
   - work_or_schedule_conflict
   - transport_or_travel_issue
   - felt_better_or_symptoms_resolved
   - financial_or_fee_concern
   - clinic_wait_time_or_rescheduled
   - family_or_personal_emergency
   - other
2. confidence: a score between 0.0 and 1.0 showing how clearly the note states this reason.
3. summary: a concise summary of the note in at most 25 words.
4. recommended_action: one practical follow-up step for the clinic."""
  format = "Return a valid JSON object matching the required schema."
  tone = "Objective, practical, and concise."

  return f"""Context: {context}

Role: {role}

Action:
{action}

Format: {format}

Tone: {tone}

Call-centre note: "{note_text}"
"""


# 5. Analyze a single note and return a dictionary
def analyse_note(note_text: str) -> dict:
  prompt = build_craft_prompt(note_text)

  response = client.models.generate_content(
      model="gemini-3.5-flash",
      contents=prompt,
      config=types.GenerateContentConfig(
          system_instruction=SYSTEM_TEXT,
          temperature=0,
          max_output_tokens=1000,
          response_mime_type="application/json",
          response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(thinking_level="minimal"),
      ),
  )

  return json.loads(response.text)


# 6. Load /content/followup_notes.csv and test on note N001
notes_df = pd.read_csv("/content/flollowup_notes.csv")
n001_row = notes_df[notes_df.iloc[:, 0] == "N001"].iloc[0]
n001_text = n001_row.iloc[-1]

print("Note N001 text:", n001_text)
parsed_result = analyse_note(n001_text)
print("\nParsed dictionary:")
print(parsed_result)

 Task 4.2 — Validate and process the 10 notes

In [ ]:

# (a) Define Pydantic model and allowed root_cause values
ALLOWED_ROOT_CAUSES = {
    "forgot_or_no_reminder",
    "work_or_schedule_conflict",
    "transport_or_travel_issue",
    "felt_better_or_symptoms_resolved",
    "financial_or_fee_concern",
    "clinic_wait_time_or_rescheduled",
    "family_or_personal_emergency",
    "other",
}


class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3)
  confidence: float = Field(ge=0, le=1)
  summary: str = Field(min_length=5)
  recommended_action: str = Field(min_length=5)


def is_valid_insight(data: dict) -> bool:
  insight = FollowupInsight(**data)
  return insight.root_cause in ALLOWED_ROOT_CAUSES


# Helper to capture raw JSON text along with parsed dict without modifying reply
def call_analyse_note_raw(note_text: str):
  prompt = build_craft_prompt(note_text)
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt,
      config=types.GenerateContentConfig(
          system_instruction=SYSTEM_TEXT,
          temperature=0,
          max_output_tokens=1000,
          response_mime_type="application/json",
          response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(thinking_level="minimal"),
      ),
  )
  raw_text = response.text or ""
  parsed = json.loads(raw_text)
  return parsed, raw_text


# (b) Read followup_notes.csv with pandas and loop over the first 10 rows
notes_df = pd.read_csv("/content/flollowup_notes.csv")
records = []

for _, row in notes_df.head(10).iterrows():
  note_id = row["note_id"] if "note_id" in row.index else row.iloc[0]
  appointment_id = (
      row["appointment_id"]
      if "appointment_id" in row.index
      else (row.iloc[1] if len(row) > 1 else "")
  )
  note_text = row["note_text"] if "note_text" in row.index else row.iloc[-1]

  # Check NaN first, then whitespace-only strings
  if pd.isna(note_text) or str(note_text).strip() == "":
    records.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": "other",
        "confidence": 0.0,
        "summary": "No note recorded",
        "recommended_action": "Re-call patient",
        "validation_status": "skipped_blank",
        "raw_response": "",
    })
    continue

  raw_response = ""
  parsed_dict = {}
  validation_status = "invalid"

  try:
    parsed_dict, raw_response = call_analyse_note_raw(str(note_text))
    if isinstance(parsed_dict, dict) and is_valid_insight(parsed_dict):
      validation_status = "valid"
    else:
      validation_status = "invalid"
  except (ValidationError, json.JSONDecodeError, TypeError, Exception):
    validation_status = "invalid"

  if not isinstance(parsed_dict, dict):
    parsed_dict = {}

  records.append({
      "note_id": note_id,
      "appointment_id": appointment_id,
      "root_cause": parsed_dict.get("root_cause", ""),
      "confidence": parsed_dict.get("confidence", ""),
      "summary": parsed_dict.get("summary", ""),
      "recommended_action": parsed_dict.get("recommended_action", ""),
      "validation_status": validation_status,
      "raw_response": raw_response,
  })

  # (c) Pause between API calls to respect free-tier rate limit
  time.sleep(13)

# (d) Save notes_analysed.csv with exact columns and un-repaired values
columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response",
]
notes_analysed_df = pd.DataFrame(records, columns=columns)
notes_analysed_df.to_csv("/content/notes_analysed.csv", index=False)

print("Saved /content/notes_analysed.csv")
display(notes_analysed_df)

Saved /content/notes_analysed.csv


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,,,,,invalid,
1,N002,A0897,,,,,invalid,
2,N003,A0526,,,,,invalid,
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,,,,,invalid,
5,N006,A0507,,,,,invalid,
6,N007,A0333,,,,,invalid,
7,N008,A0222,,,,,invalid,
8,N009,A0977,,,,,invalid,
9,N010,A0596,,,,,invalid,


Task 4.3 — Break it on purpose

In [ ]:
# --- (a) Test Pydantic ValidationError with two deliberately bad replies ---
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,  # Violates le=1 constraint
    "summary": "Patient forgot the scheduled visit.",
    "recommended_action": "Send reminder call before next visit.",
}

bad_reply_2 = {
    "root_cause": "other",
    "confidence": 0.6,
    "summary": "Note is unclear about absence.",
    # Missing recommended_action required field
}

print("--- (a) ValidationError Tests ---")
try:
  # Unpack dictionary keys into FollowupInsight to trigger Pydantic validation
  FollowupInsight(**bad_reply_1)
except ValidationError as e:
  print("\nBad reply 1 (confidence = 1.7) error:")
  print(e)

try:
  FollowupInsight(**bad_reply_2)
except ValidationError as e:
  print("\nBad reply 2 (missing recommended_action) error:")
  print(e)


# --- (b) Compare V1 vs V2 prompts on N007 and N009 at temperature=0 ---
def classify_with_prompt(prompt_text: str) -> dict:
  # Call Gemini with identical schema, JSON mode, and temperature=0
  response = client.models.generate_content(
      model="gemini-3.5-flash",
      contents=prompt_text,
      config=types.GenerateContentConfig(
          system_instruction=SYSTEM_TEXT,
          temperature=0,
          max_output_tokens=1000,
          response_mime_type="application/json",
          response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(thinking_level="minimal"),
      ),
  )
  return json.loads(response.text)


notes_df = pd.read_csv("/content/flollowup_notes.csv")
target_ids = ["N007", "N009"]
target_rows = notes_df[notes_df.iloc[:, 0].isin(target_ids)]

comparison_rows = []
v1_calls = 0
v2_calls = 0

for _, row in target_rows.iterrows():
  note_id = row["note_id"] if "note_id" in row.index else row.iloc[0]
  note_text = row["note_text"] if "note_text" in row.index else row.iloc[-1]

  # V1 prompt: minimal instruction only
  v1_prompt = f'Classify this note\n\nCall-centre note: "{note_text}"'
  res_v1 = classify_with_prompt(v1_prompt)
  v1_calls += 1
  time.sleep(13)

  # V2 prompt: CRAFT prompt with allowed values and 'other' fallback rule
  v2_prompt = build_craft_prompt(note_text)
  res_v2 = classify_with_prompt(v2_prompt)
  v2_calls += 1
  time.sleep(13)

  for variant, parsed in [("V1", res_v1), ("V2", res_v2)]:
    rc = parsed.get("root_cause", "")
    in_allowed = rc in ALLOWED_ROOT_CAUSES
    comparison_rows.append({
        "note_id": note_id,
        "prompt": variant,
        "root_cause": rc,
        "in_allowed_list": in_allowed,
        "confidence": parsed.get("confidence", ""),
        "summary": parsed.get("summary", ""),
    })

comparison_table = pd.DataFrame(comparison_rows)
print("\n--- (b) V1 vs V2 Comparison Table (N007 & N009) ---")
display(comparison_table)

print("\nDeployment Recommendation:")
print(
    "Deploy V2 (CRAFT prompt): it lists the allowed root causes clearly and"
    " instructs the model to choose 'other' instead of guessing on Hinglish"
    " or ambiguous notes."
)


# --- (c) Count and print total API calls across Tasks 4.1 - 4.3 ---
calls_4_1 = 1

if "notes_analysed_df" in globals():
  calls_4_2 = int(
      (notes_analysed_df["validation_status"] != "skipped_blank").sum()
  )
else:
  calls_4_2 = 9

calls_4_3 = v1_calls + v2_calls
total_api_calls = calls_4_1 + calls_4_2 + calls_4_3

print("\n--- (c) Total API Calls Across Tasks 4.1-4.3 ---")
print(f"Task 4.1 API calls: {calls_4_1}")
print(f"Task 4.2 API calls: {calls_4_2}")
print(f"Task 4.3 API calls: {calls_4_3} (V1: {v1_calls}, V2: {v2_calls})")
print(f"Total API calls (4.1–4.3): {total_api_calls}")

--- (a) ValidationError Tests ---

Bad reply 1 (confidence = 1.7) error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Bad reply 2 (missing recommended_action) error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...unclear about absence.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

--- (b) V1 vs V2 Comparison Table (N007 & N009) ---


,note_id,prompt,root_cause,in_allowed_list,confidence,summary
0,N007,V1,Lapse in patient memory coupled with missed or...,False,0.92,The patient reported forgetting their appointm...
1,N007,V2,forgot_or_no_reminder,True,0.95,Patient missed appointment because they forgot...
2,N009,V1,Unreachable Patient - Phone Switched Off,False,0.98,The call center attempted to contact the patie...
3,N009,V2,other,True,1.00,Patient was unreachable after three call attem...



Deployment Recommendation:
Deploy V2 (CRAFT prompt): it lists the allowed root causes clearly and instructs the model to choose 'other' instead of guessing on Hinglish or ambiguous notes.

--- (c) Total API Calls Across Tasks 4.1-4.3 ---
Task 4.1 API calls: 1
Task 4.2 API calls: 9
Task 4.3 API calls: 4 (V1: 2, V2: 2)
Total API calls (4.1–4.3): 14


Task 4.4 — Recommendation and GitHub

In [ ]:
# 1. Load analysed follow-up notes and cleaned appointments
notes_analysed = pd.read_csv("/content/notes_analysed.csv")
appointments_clean = pd.read_csv("/content/appointments_clean.csv")

# 2. Merge on appointment_id (inner merge keeps matched follow-up appointments)
merged_notes_appt = pd.merge(
    notes_analysed,
    appointments_clean,
    on="appointment_id",
    how="inner",
)

print("Merged shape:", merged_notes_appt.shape)
display(
    merged_notes_appt[
        ["note_id", "appointment_id", "root_cause", "reminder_sent", "status"]
    ]
)

# 3. Cross-tabulate root_cause against reminder_sent (including missing/blank reminder_sent if any)
crosstab_rc_reminder = pd.crosstab(
    merged_notes_appt["root_cause"],
    merged_notes_appt["reminder_sent"],
    dropna=False,
)

print("\nCross-tabulation of root_cause vs. reminder_sent:")
display(crosstab_rc_reminder)

Merged shape: (10, 23)


,note_id,appointment_id,root_cause,reminder_sent,status
0,N001,A0872,NaN,No,No-Show
1,N002,A0897,NaN,No,No-Show
2,N003,A0526,NaN,Yes,No-Show
3,N004,A0926,other,No,No-Show
4,N005,A0534,NaN,Yes,No-Show
5,N006,A0507,NaN,Yes,No-Show
6,N007,A0333,NaN,No,No-Show
7,N008,A0222,NaN,Yes,No-Show
8,N009,A0977,NaN,Yes,No-Show
9,N010,A0596,NaN,No,No-Show



Cross-tabulation of root_cause vs. reminder_sent:


reminder_sent,No,Yes
root_cause,,
other,1,0
NaN,4,5


In [ ]:
https://github.com/shrutika01611/careloop-noshow-guard

SyntaxError: invalid syntax (574431986.py, line 1)

In [ ]:
readme_content = """# CareLoop Clinics — No-Show Guard

**GitHub Repository:** https://github.com/shrutika01611/careloop-noshow-guard

## Summary
1. **Part 1 (Data Cleaning & Enrichment):** Cleaned `appointments.csv` from **1,020** raw rows down to **990** valid rows (**218** no-shows overall, **22.0%** no-show rate) and enriched all **8** outpatient clinics with live weather temperatures.
2. **Part 2 (Exploratory & SQL Analysis):** Across the **8** clinics, no-show rates ranged from **13.8%** (Vaishali Nagar Clinic, Jaipur) up to **33.3%** (Gomti Nagar Care Point, Lucknow).
3. **Part 3 (Predictive Modeling):** Selected a `StandardScaler + LogisticRegression` pipeline (`cv_auc` = **0.7737**, test ROC-AUC = **0.7889**) using a **0.35** risk threshold instead of **0.50**.
4. **Part 4 (GenAI Follow-Up Notes & Validation):** Validated structured call-centre note extraction (`FollowupInsight` Pydantic schema + allowed `root_cause` list), skipped blank notes cleanly, and used a CRAFT prompt with explicit allowed labels and an `other` fallback.

## Recommendation
To reduce missed outpatient slots at CareLoop Clinics—where **218** of **990** cleaned bookings (**22.0%**) resulted in no-shows and clinic no-show rates reach **33.3%** in Lucknow (Gomti Nagar Care Point) and **27.3%** in Pune (Hadapsar Care Centre)—CareLoop should deploy the evening-before `LogisticRegression` risk scorer at a **0.35** probability threshold (achieving **0.79** test ROC-AUC and **0.57** test recall, flagging **49** of **198** test appointments) and prioritize outreach for high-risk bookings with long `lead_days` (mean **11.82** days; **29.9%** of bookings over **14** days), prior no-shows, or unconfirmed reminders. Pairing evening calls with structured call-centre `root_cause` tracking (such as `forgot_or_no_reminder` vs. rescheduling/travel issues) lets staff target reminder timing and slot backfill where missed visits are highest while keeping call volume manageable.

## Model Choice & Threshold
We chose the `StandardScaler + LogisticRegression` pipeline over the unconstrained decision tree (which overfit with **1.00** train accuracy vs. **0.67** test accuracy), depth-3 tree (**0.6784** 5-fold CV ROC-AUC), and untuned/tuned random forests (**0.7466** and **0.7669** 5-fold CV ROC-AUC), because it achieved the highest 5-fold cross-validated training ROC-AUC (**0.7737**) and test ROC-AUC (**0.7889**, rounded to **0.79**). We chose threshold **0.35** over **0.50** because on out-of-fold training predictions it raised F1 from **0.3471** to **0.4892** and recall from **0.2485** to **0.4675** (reaching **0.57** recall and **0.57** precision on the test set with **49** flagged appointments), aligning with clinic operations where missing a no-show wastes a doctor slot while an extra call takes only a minute.

## Limitations
- **The 10 notes are an illustration and cannot show the real reasons clinic-wide:** Analyzing only the first **10** follow-up notes (including blank/skipped rows) provides a small sample and cannot represent root-cause proportions across all missed visits.
- **The cleaning statistics came from all rows before the split:** Imputing missing `age` and `distance_km` with medians and computing IQR outlier fences before splitting can introduce minor preprocessing leakage.
- **The data are observational, so associations are not causes:** Features such as `reminder_sent` and `lead_days` show statistical associations with no-shows in historical records rather than causal proof from a randomized intervention.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md sucessfully created!")